# Semi-Supervised Learning: MixMatch (PyTorch)

**Buku:** *Deep Learning dengan Python* — Bab 13, Semi-Supervised Learning

MixMatch menebak label data tak berlabel dari rata-rata K augmentasi, mempertajamnya (*sharpening*), lalu mencampur data berlabel dan tak berlabel dengan MixUp. Loss tak berlabel berupa galat kuadrat (L2) yang bobotnya dinaikkan bertahap.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Notebook ini memakai **PyTorch** murni (sesuai kerangka kode Bab 12–13).

**Perkiraan waktu (GPU T4):** ±45–60 menit untuk 10.000 langkah (makalah asli: 2²⁰ langkah)

**Isi notebook:**
- Tebakan label: rata-rata prediksi K = 2 augmentasi + *sharpening* dengan temperature T.
- MixUp dengan λ' = max(λ, 1 − λ) agar setiap campuran tetap dekat dengan sampel aslinya.
- Loss: *cross-entropy* (berlabel) + λ_u · L2 (tak berlabel) dengan *ramp-up* linear.
- Optimizer Adam + *weight decay* gaya MixMatch + EMA, seperti implementasi resmi.
- Komponen data, model, dan pelaporan sama dengan notebook FixMatch, sehingga hasilnya bisa dibandingkan langsung.

In [ ]:
import torch, torchvision, numpy as np
perangkat = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__, "| perangkat:", perangkat)

In [ ]:
N_BERLABEL = 4000       # 400 per kelas; coba 250 atau 40 (seperti di makalah)

B = 64                  # ukuran batch berlabel = tak berlabel
K = 2                   # jumlah augmentasi untuk menebak label
SUHU = 0.5              # temperature sharpening
ALPHA = 0.75            # parameter distribusi Beta untuk MixUp
LAMBDA_U = 75.0         # bobot loss tak berlabel (nilai makalah untuk CIFAR-10)
LR = 0.002
LANGKAH = 10000
RAMPUP = min(16384, LANGKAH // 3)
EVAL_SETIAP = 500
LANGKAH_SUP = 3000

## 1. Data, model, dan utilitas

In [ ]:
import copy, math, time
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

latih_mentah = torchvision.datasets.CIFAR10("data", train=True, download=True)
uji_mentah = torchvision.datasets.CIFAR10("data", train=False, download=True)
X, Y = latih_mentah.data, np.array(latih_mentah.targets)
KELAS = latih_mentah.classes

def bagi_data(Y, n_berlabel, seed=0):
    """Protokol standar: n_berlabel/10 contoh berlabel per kelas; SELURUH data latih
    (termasuk yang berlabel) dipakai sebagai data tak berlabel."""
    rng = np.random.default_rng(seed)
    berlabel = []
    for c in range(10):
        berlabel += list(rng.choice(np.where(Y == c)[0], n_berlabel // 10, replace=False))
    return np.array(berlabel), np.arange(len(Y))

idx_l, idx_u = bagi_data(Y, N_BERLABEL)
print(f"Berlabel: {len(idx_l)} | tak berlabel: {len(idx_u)}")
print("Berlabel per kelas:", np.bincount(Y[idx_l], minlength=10))

RERATA, SIMPANGAN = (0.4914, 0.4822, 0.4465), (0.2471, 0.2435, 0.2616)
normalisasi = [T.ToTensor(), T.Normalize(RERATA, SIMPANGAN)]
lemah = T.Compose([T.RandomHorizontalFlip(), T.RandomCrop(32, padding=4, padding_mode="reflect")] + normalisasi)
kuat = T.Compose([T.RandomHorizontalFlip(), T.RandomCrop(32, padding=4, padding_mode="reflect"),
                  T.RandAugment(num_ops=2, magnitude=10)] + normalisasi +
                 [T.RandomErasing(p=1.0, scale=(0.02, 0.25), ratio=(1, 1), value=0)])   # ~Cutout
polos = T.Compose(normalisasi)

class Subset(Dataset):
    """Subset CIFAR dengan transformasi; transformasi boleh mengembalikan tuple banyak pandangan."""
    def __init__(self, data, target, idx, transformasi):
        self.data, self.target, self.idx, self.t = data, target, idx, transformasi
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        return self.t(Image.fromarray(self.data[j])), self.target[j]

class BanyakPandangan:
    def __init__(self, *transformasi):
        self.daftar = transformasi
    def __call__(self, x):
        return tuple(t(x) for t in self.daftar)

loader_uji = DataLoader(Subset(uji_mentah.data, np.array(uji_mentah.targets),
                               np.arange(10000), polos), batch_size=512, num_workers=2)

def siklus(loader):
    while True:
        for batch in loader:
            yield batch

In [ ]:
class BlokDasar(nn.Module):
    """Blok residual pre-activation (BN -> LeakyReLU -> Conv) x 2."""
    def __init__(self, masuk, keluar, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(masuk, momentum=0.001)
        self.conv1 = nn.Conv2d(masuk, keluar, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(keluar, momentum=0.001)
        self.conv2 = nn.Conv2d(keluar, keluar, 3, 1, 1, bias=False)
        self.akt = nn.LeakyReLU(0.1)
        self.identitas = masuk == keluar and stride == 1
        self.pintas = None if self.identitas else nn.Conv2d(masuk, keluar, 1, stride, bias=False)

    def forward(self, x):
        o = self.akt(self.bn1(x))
        pintas = x if self.identitas else self.pintas(o)
        o = self.conv2(self.akt(self.bn2(self.conv1(o))))
        return o + pintas

class WideResNet(nn.Module):
    """WRN-28-2 (±1,5 juta parameter), arsitektur standar MixMatch/FixMatch untuk CIFAR-10."""
    def __init__(self, n_kelas=10, kedalaman=28, lebar=2):
        super().__init__()
        n = (kedalaman - 4) // 6
        k = [16, 16 * lebar, 32 * lebar, 64 * lebar]
        grup = lambda masuk, keluar, s: nn.Sequential(
            *[BlokDasar(masuk if i == 0 else keluar, keluar, s if i == 0 else 1) for i in range(n)])
        self.awal = nn.Conv2d(3, k[0], 3, 1, 1, bias=False)
        self.badan = nn.Sequential(grup(k[0], k[1], 1), grup(k[1], k[2], 2), grup(k[2], k[3], 2))
        self.bn = nn.BatchNorm2d(k[3], momentum=0.001)
        self.akt = nn.LeakyReLU(0.1)
        self.fc = nn.Linear(k[3], n_kelas)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="leaky_relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        o = self.akt(self.bn(self.badan(self.awal(x))))
        return self.fc(F.adaptive_avg_pool2d(o, 1).flatten(1))

class EMA:
    """Rata-rata bergerak eksponensial dari bobot; model EMA dipakai untuk evaluasi."""
    def __init__(self, model, peluruhan=0.999):
        self.model = copy.deepcopy(model).eval()
        self.peluruhan = peluruhan
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def perbarui(self, model):
        for e, m in zip(self.model.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point:
                e.mul_(self.peluruhan).add_(m.detach(), alpha=1 - self.peluruhan)
            else:
                e.copy_(m)

@torch.no_grad()
def evaluasi(model, loader=loader_uji):
    model.eval()
    prediksi, benar = [], []
    for x, y in loader:
        prediksi.append(model(x.to(perangkat)).argmax(1).cpu()); benar.append(y)
    prediksi, benar = torch.cat(prediksi).numpy(), torch.cat(benar).numpy()
    per_kelas = np.array([(prediksi[benar == c] == c).mean() for c in range(10)])
    return (prediksi == benar).mean(), per_kelas

model = WideResNet().to(perangkat)
print(f"Parameter: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from torch.utils.data import RandomSampler

def buat_optimizer(model, lr=0.03, wd=5e-4):
    """SGD Nesterov; weight decay tidak dikenakan pada bias dan parameter BatchNorm."""
    decay = [p for p in model.parameters() if p.ndim > 1]
    tanpa = [p for p in model.parameters() if p.ndim <= 1]
    return torch.optim.SGD([{"params": decay, "weight_decay": wd}, {"params": tanpa, "weight_decay": 0.0}],
                           lr=lr, momentum=0.9, nesterov=True)

def jadwal_kosinus(opt, total):
    return torch.optim.lr_scheduler.LambdaLR(opt, lambda k: math.cos(7 * math.pi * k / (16 * total)))

def loader_acak(ds, batch, langkah, pekerja=2):
    """Sampling dengan pengembalian: jumlah batch = langkah, berapa pun ukuran dataset."""
    return DataLoader(ds, batch_size=batch, num_workers=pekerja, drop_last=True, pin_memory=True,
                      sampler=RandomSampler(ds, replacement=True, num_samples=batch * langkah))

pakai_amp = perangkat == "cuda"
def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=pakai_amp)

def latih_terawasi(langkah, batch=64):
    """Baseline: WRN-28-2 hanya dari data berlabel, dengan augmentasi lemah."""
    model_s = WideResNet().to(perangkat)
    opt, ema_s = buat_optimizer(model_s), EMA(model_s)
    jadwal = jadwal_kosinus(opt, langkah)
    skala = torch.amp.GradScaler(enabled=pakai_amp)
    model_s.train()
    for x, y in loader_acak(Subset(X, Y, idx_l, lemah), batch, langkah):
        with autocast():
            loss = F.cross_entropy(model_s(x.to(perangkat)), y.to(perangkat))
        opt.zero_grad(set_to_none=True)
        skala.scale(loss).backward(); skala.step(opt); skala.update()
        jadwal.step(); ema_s.perbarui(model_s)
    return evaluasi(ema_s.model)

## 2. Baseline: hanya data berlabel

In [ ]:
akurasi_sup, per_kelas_sup = latih_terawasi(LANGKAH_SUP)
print(f"Hanya {len(idx_l)} label: akurasi uji {akurasi_sup:.2%}")

## 3. Dua bahan inti: *sharpening* dan MixUp

In [ ]:
def pertajam(p, suhu=SUHU):
    p = p ** (1 / suhu)
    return p / p.sum(dim=1, keepdim=True)

p = torch.tensor([[0.5, 0.3, 0.2]])
for t in [1.0, 0.5, 0.25]:
    print(f"T = {t:4.2f}: {pertajam(p, t).numpy().round(3)}")

def mixup(x, y, alpha=ALPHA):
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)                  # campuran tetap 'milik' sampel pertama
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]

## 4. Pelatihan MixMatch

In [ ]:
ds_l = Subset(X, Y, idx_l, lemah)
ds_u = Subset(X, Y, idx_u, BanyakPandangan(*[lemah] * K))
loader_l = loader_acak(ds_l, B, LANGKAH)
loader_u = loader_acak(ds_u, B, LANGKAH)

model = WideResNet().to(perangkat)
ema = EMA(model)
opt = torch.optim.Adam(model.parameters(), lr=LR)
skala = torch.amp.GradScaler(enabled=pakai_amp)

log = {"langkah": [], "akurasi": [], "rasio_masker": [], "presisi_semu": []}
akum = {"yakin": 0.0, "benar": 0.0, "n": 0}
model.train(); t0 = time.time()

for k, ((x_l, y_l), (pandangan_u, y_u)) in enumerate(zip(loader_l, loader_u), start=1):
    x_l = x_l.to(perangkat, non_blocking=True)
    y_l1h = F.one_hot(y_l, 10).float().to(perangkat)
    pandangan_u = [u.to(perangkat, non_blocking=True) for u in pandangan_u]

    # (1) Tebak label: rata-rata K augmentasi, lalu pertajam
    with torch.no_grad():
        with autocast():
            logit_u = model(torch.cat(pandangan_u)).float()
        p = torch.softmax(logit_u, dim=1).view(K, B, 10).mean(0)
        q = pertajam(p)

    # (2) MixUp seluruh batch: [berlabel, u_1, ..., u_K]
    semua_x = torch.cat([x_l] + pandangan_u)
    semua_y = torch.cat([y_l1h] + [q] * K)
    x_campur, y_campur = mixup(semua_x, semua_y)

    # (3) Loss: CE untuk bagian berlabel, L2 untuk bagian tak berlabel
    with autocast():
        logit = model(x_campur).float()
    loss_x = -(F.log_softmax(logit[:B], dim=1) * y_campur[:B]).sum(1).mean()
    loss_u = ((torch.softmax(logit[B:], dim=1) - y_campur[B:]) ** 2).mean()
    w = LAMBDA_U * min(1.0, k / RAMPUP)                     # ramp-up linear
    loss = loss_x + w * loss_u

    opt.zero_grad(set_to_none=True)
    skala.scale(loss).backward(); skala.step(opt); skala.update()
    with torch.no_grad():                                    # weight decay ala MixMatch
        for prm in model.parameters():
            prm.mul_(1 - 0.02 * LR)
    ema.perbarui(model)

    keyakinan, tebakan = q.max(1)
    akum["yakin"] += (keyakinan >= 0.95).float().sum().item(); akum["n"] += B
    akum["benar"] += (tebakan.cpu() == y_u).float().sum().item()

    if k % EVAL_SETIAP == 0 or k == LANGKAH:
        akurasi, _ = evaluasi(ema.model); model.train()
        log["langkah"].append(k); log["akurasi"].append(akurasi)
        log["rasio_masker"].append(akum["yakin"] / akum["n"])
        log["presisi_semu"].append(akum["benar"] / akum["n"])
        print(f"langkah {k:5d} | loss_x {loss_x.item():.3f} loss_u {loss_u.item():.4f} w {w:5.1f} | "
              f"yakin {log['rasio_masker'][-1]:.1%} akurasi tebakan {log['presisi_semu'][-1]:.1%} | "
              f"akurasi uji {akurasi:.2%} | {time.time() - t0:.0f} dtk")
        akum = {"yakin": 0.0, "benar": 0.0, "n": 0}

torch.save(ema.model.state_dict(), "mixmatch_wrn28_2.pt")

## 5. Hasil

In [ ]:
import matplotlib.pyplot as plt

def laporan(log, per_kelas_ssl, per_kelas_sup, judul):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(log["langkah"], log["akurasi"], "o-"); ax[0].set(xlabel="Langkah", ylabel="Akurasi uji (EMA)", title=judul)
    ax[1].plot(log["langkah"], log["rasio_masker"], label="proporsi yakin (≥ ambang)"); ax[1].plot(log["langkah"], log["presisi_semu"], label="presisi label semu")
    ax[1].set(xlabel="Langkah", title="Kualitas label semu"); ax[1].legend()
    lebar = 0.4; posisi = np.arange(10)
    ax[2].bar(posisi - lebar / 2, per_kelas_sup, lebar, label="hanya berlabel")
    ax[2].bar(posisi + lebar / 2, per_kelas_ssl, lebar, label=judul)
    ax[2].set_xticks(posisi, KELAS, rotation=60); ax[2].set(ylabel="Akurasi per kelas"); ax[2].legend()
    plt.tight_layout(); plt.show()
    print(f"Akurasi rata-rata per kelas (balanced accuracy): hanya berlabel {per_kelas_sup.mean():.2%} | {judul} {per_kelas_ssl.mean():.2%}")

akurasi_ssl, per_kelas_ssl = evaluasi(ema.model)
print(f"Hanya berlabel: {akurasi_sup:.2%} | MixMatch: {akurasi_ssl:.2%}")
laporan(log, per_kelas_ssl, per_kelas_sup, "MixMatch")

**Catatan:** MixMatch tidak memakai ambang, sehingga semua tebakan ikut melatih (dengan bobot lunak). Kurva "yakin ≥ 0,95" hanya untuk pemantauan dan perbandingan dengan FixMatch.

**Eksperimen:**
1. Ablasi seperti di makalah: matikan *sharpening* (`SUHU = 1.0`), matikan MixUp (`ALPHA` sangat kecil, misalnya `1e-6`), atau pakai `K = 1`. Komponen mana yang paling berpengaruh?
2. Ganti loss L2 dengan *cross-entropy* pada `loss_u`. Mengapa makalah memilih L2?